In [ ]:

# ==========================================================
# Cell 1 - Imports and Data Loading
# ==========================================================

import sys
from pathlib import Path

import pandas as pd
import numpy as np

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data_preparation import (
    load_and_clean_data,
    separate_features_target
)

from src.validation import create_train_test_split

from src.feature_engineering import TelcoFeatureEngineer

from src.preprocessing import build_preprocessor

# Load dataset
df = load_and_clean_data(project_root)

X, y = separate_features_target(df)

X_train, X_test, y_train, y_test = (
    create_train_test_split(X, y)
)

print("All imports successful!")
print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)


All imports successful!
Training shape: (5634, 19)
Testing shape: (1409, 19)


##### Cell 1 — Load the shared training data

In [ ]:

# ==========================================================
# Feature engineering verification
# ==========================================================

from src.feature_engineering import TelcoFeatureEngineer

baseline_engineer = TelcoFeatureEngineer(
    add_engineered_features=False
)

engineered_engineer = TelcoFeatureEngineer(
    add_engineered_features=True
)

X_train_baseline = baseline_engineer.fit_transform(X_train)

X_train_engineered = engineered_engineer.fit_transform(X_train)

print("Original features:", X_train.shape[1])
print("Baseline features:", X_train_baseline.shape[1])
print("Engineered features:", X_train_engineered.shape[1])

print("\nNew features:")
print(
    sorted(
        set(X_train_engineered.columns)
        - set(X_train_baseline.columns)
    )
)

display(
    X_train_engineered[
        ["tenure", "TenureGroup", "ServiceCount"]
    ].head(10)
)


Original features: 19
Baseline features: 19
Engineered features: 21

New features:
['ServiceCount', 'TenureGroup']


,tenure,TenureGroup,ServiceCount
3738,35,25-48,4
3151,15,13-24,3
4860,13,13-24,4
3867,26,25-48,6
3810,1,0-12,2
2666,66,49-72,7
2645,1,0-12,1
189,40,25-48,4
5767,65,49-72,4
2257,60,49-72,7


##### Cell 2 — Test baseline preprocessing with scaling

In [ ]:


baseline_preprocessor = build_preprocessor(
    scale_numeric=True,
    add_engineered_features=False
)

X_train_processed = baseline_preprocessor.fit_transform(
    X_train
)

feature_names = (
    baseline_preprocessor.get_feature_names_out()
)

print("Original shape:", X_train.shape)
print("Processed shape:", X_train_processed.shape)

print("Number of output features:", len(feature_names))

print("\nFirst 20 feature names:")
print(feature_names[:20])

print("\nMissing values after preprocessing:")
print(np.isnan(X_train_processed).sum())


Original shape: (5634, 19)
Processed shape: (5634, 45)
Number of output features: 45

First 20 feature names:
['tenure' 'MonthlyCharges' 'TotalCharges' 'SeniorCitizen' 'gender_Female'
 'gender_Male' 'Partner_No' 'Partner_Yes' 'Dependents_No' 'Dependents_Yes'
 'PhoneService_No' 'PhoneService_Yes' 'MultipleLines_No'
 'MultipleLines_No phone service' 'MultipleLines_Yes'
 'InternetService_DSL' 'InternetService_Fiber optic' 'InternetService_No'
 'OnlineSecurity_No' 'OnlineSecurity_No internet service']

Missing values after preprocessing:
0


##### Cell 3 — Test engineered preprocessing

In [ ]:


engineer = TelcoFeatureEngineer(
    add_engineered_features=True
)

X_train_engineered = engineer.fit_transform(X_train)

engineered_preprocessor = build_preprocessor(
    scale_numeric=True,
    add_engineered_features=True
)

X_train_engineered_processed = (
    engineered_preprocessor.fit_transform(
        X_train_engineered
    )
)

engineered_feature_names = (
    engineered_preprocessor.get_feature_names_out()
)

print("Engineered input shape:", X_train_engineered.shape)

print(
    "Engineered processed shape:",
    X_train_engineered_processed.shape
)

print(
    "Number of processed features:",
    len(engineered_feature_names)
)

print(
    "Missing values:",
    np.isnan(X_train_engineered_processed).sum()
)

print(
    "ServiceCount included:",
    "ServiceCount" in engineered_feature_names
)

print(
    "TenureGroup columns:",
    [
        name for name in engineered_feature_names
        if name.startswith("TenureGroup_")
    ]
)


Engineered input shape: (5634, 21)
Engineered processed shape: (5634, 50)
Number of processed features: 50
Missing values: 0
ServiceCount included: True
TenureGroup columns: ['TenureGroup_0-12', 'TenureGroup_13-24', 'TenureGroup_25-48', 'TenureGroup_49-72']


##### Cell 4 — Verify scaling

In [ ]:

# Responsibility: IT24102310 - Scaling

scaled_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

display(
    scaled_df[
        ["tenure", "MonthlyCharges", "TotalCharges"]
    ].agg(["mean", "std"]).round(4)
)

print(
    "All processed values finite:",
    np.isfinite(X_train_processed).all()
)


,tenure,MonthlyCharges,TotalCharges
mean,-0.0000,-0.0000,0.0000
std,1.0001,1.0001,1.0001


All processed values finite: True
